In [ ]:
import time
import copy
from IPython.display import clear_output

# ── Colori ────────────────────────────────────────────────────────────────────
ANSI_COLORS = ["\033[91m", "\033[92m", "\033[93m", "\033[94m", "\033[95m", "\033[96m"]
YELLOW_STATION = "\033[1;93m"
RESET = "\033[0m"

def init_key_colors(stanze):
    keys = [item for item in stanze if item.startswith("key")]
    return {key: ANSI_COLORS[i % len(ANSI_COLORS)] for i, key in enumerate(keys)}

def colored_key(key_name, key_colors):
    color = key_colors.get(key_name, "")
    return f"{color}k{RESET}"

def door_color(door_name, key_colors):
    key_name = door_name.replace("door_", "key_")
    return key_colors.get(key_name, "")

# ── Azioni ────────────────────────────────────────────────────────────────────
def get(hand, action, stanze, bal, mani):
    stanze[bal] = ""
    if hand == 0:
        mani[0] = action[12:] if action.startswith("get_left_box") else action[9:]
    else:
        mani[1] = action[13:] if action.startswith("get_right_box") else action[10:]

def put(hand, action, stanze, bal, mani):
    if hand == 0:
        mani[0] = "o"
        stanze[bal] = action[9:]
    else:
        mani[1] = "o"
        stanze[bal] = action[10:]

def swap(action, stanze, bal, mani):
    if action.startswith("swap_left"):
        old_hand = action[10:]
        get(0, "get_left_" + stanze[bal], stanze, bal, mani)
        stanze[bal] = old_hand
    else:
        old_hand = action[11:]
        get(1, "get_right_" + stanze[bal], stanze, bal, mani)
        stanze[bal] = old_hand

# ── Funzione di stampa  ───────────────────────────────
def draw_grid(azione_corrente, stanze, bal, batteria, porte, mani, key_colors):
    clear_output(wait=True)
    print(porte)
    print()
    print("Batteria: ", batteria)
    print("Azione eseguita: ", azione_corrente)
    print()

    appStanze = len(stanze)
    appBal    = bal
    c         = RESET

    for k in range(2):
        if appStanze % 2 == 1:
            print("      ", end="")
        for j in range(appStanze // 2):
            idx = abs(k * (len(stanze) - 2) - j) + k
            if (j + 1) == appStanze // 2 and k == 1:
                idx -= 1
            a = idx
            b = (idx + 1) % len(stanze)
            if idx == 3 or (k == 1 and b == 3):
                c = YELLOW_STATION
            else:
                c = RESET
            if len(stanze) % 2 == 0:
                print(f"{c}+-", end="")
                if k == 1 and (j == 0 or j == appStanze // 2 - 1):
                    print(f"{c}---{RESET}", end="")
                    color = door_color(f"door_{min(a,b)}_{max(a,b)}", key_colors)
                    if porte[f"door_{min(a,b)}_{max(a,b)}"] == "open":
                        print(f"{color}  |{RESET}", end="")
                    else:
                        print(f"{color}___{RESET}", end="")
                    print(f"{c}---", end="")
                else:
                    print(f"{c}---------", end="")
                print(f"{c}-", end="")
            else:
                print(f"{c}+", end="")
                if k == 1 and j == appStanze // 2 - 1:
                    color = door_color(f"door_{min(a,b)}_{max(a,b)}", key_colors)
                    if porte[f"door_{min(a,b)}_{max(a,b)}"] == "open":
                        print(f"{color}    |{RESET}", end="")
                    else:
                        print(f"{color}_____{RESET}", end="")
                else:
                    print(f"{c}-----", end="")
                print(f"{c}-{RESET}" if k == 0 else f"{c}+{RESET}", end="")
                if k == 1 and j == 0:
                    color = door_color(f"door_{min(a,b)}_{max(a,b)}", key_colors)
                    if porte[f"door_{min(a,b)}_{max(a,b)}"] == "open":
                        print(f"{color}    |{RESET}", end="")
                    else:
                        print(f"{color}_____{RESET}", end="")
                else:
                    print(f"{c}-----", end="")
        print(f"{c}+")

        # parte centrale
        for l in range(5):
            if appStanze % 2 == 1:
                print("      ", end="")
            for j in range(appStanze // 2):
                idx = abs(k * (len(stanze) - 2) - j) + k + k - 1
                a = idx
                b = (idx + 1) % len(stanze)
                room_idx = abs(k * (len(stanze) - 1) - j)
                room_content = stanze[room_idx]

                if a == 3 or b == 3:
                    c = YELLOW_STATION
                else:
                    c = RESET

                if l != 2 or j == 0:
                    print(f"{c}|{RESET}", end="")
                else:
                  dcolor = door_color(f"door_{min(a,b)}_{max(a,b)}", key_colors)
                  if porte[f"door_{min(a,b)}_{max(a,b)}"] == "open":
                    print(f"{dcolor}_{RESET}", end="")
                  else:
                      print(f"{dcolor}I{RESET}", end="")
                if bal == room_idx:
                    if l == 1:
                        print("    (O)    ", end="")
                    elif l == 2:
                        l_hand = colored_key(mani[0], key_colors) if mani[0].startswith("key") else mani[0]
                        r_hand = colored_key(mani[1], key_colors) if mani[1].startswith("key") else mani[1]
                        print(f"  {l_hand}/|#|\\{r_hand}  ", end="")
                    elif l == 3:
                        print("    / \\    ", end="")
                    elif room_content.startswith("box") and l == 0:
                        print(f"    [{room_content[3:]}]    ", end="")
                    elif room_content.startswith("key") and l == 0:
                        print(f"     {colored_key(room_content, key_colors)}     ", end="")
                    else:
                        print("           ", end="")
                elif room_content.startswith("box") and l == 2:
                    print(f"    [{room_content[3:]}]    ", end="")
                elif room_content.startswith("key") and l == 2:
                    print(f"     {colored_key(room_content, key_colors)}     ", end="")
                else:
                    print("           ", end="")
            if b == 3:
                c = RESET
            print(f"{c}|")

        if k == 1:
            for j in range(appStanze // 2):
                if abs(k * (len(stanze) - 2) - j) + k + k - 1 == 3:
                    c = YELLOW_STATION
                else:
                    c = RESET
                print(f"{c}+-----------", end="")
            print(f"{c}+{RESET}")

        if appStanze % 2 == 1:
            appStanze += 1
            appBal += 1

def animate(sol, initial_state):
    action = sol.solution()
    stato_iniziale = copy.deepcopy(initial_state)
    stanze   = list(stato_iniziale["rooms"])
    bal      = stato_iniziale["robot"]
    batteria = stato_iniziale["battery"]
    porte    = stato_iniziale["doors"]
    mani     = ["o", "o"]

    key_colors = init_key_colors(stanze)

    draw_grid_fn = lambda azione_corrente: draw_grid(
        azione_corrente, stanze, bal, batteria, porte, mani, key_colors
    )

    draw_grid_fn("Stato Iniziale")
    time.sleep(1)

    for az in action:
        batteria -= 1
        if az == "goleft":
            bal = (bal - 1) % len(stanze)
        elif az == "goright":
            bal = (bal + 1) % len(stanze)
        elif az.startswith("get_left"):
            get(0, az, stanze, bal, mani)
        elif az.startswith("get_right"):
            get(1, az, stanze, bal, mani)
        elif az.startswith("put_left"):
            put(0, az, stanze, bal, mani)
        elif az.startswith("put_right"):
            put(1, az, stanze, bal, mani)
        elif az.startswith("swap"):
            swap(az, stanze, bal, mani)
        elif az == "charge":
            batteria = 10
        elif az.startswith("open"):
            porte[f"door_{az[5:]}"] = "open"

        draw_grid_fn(az)
        print("\nStanze aggiornate: ", stanze)
        print("Mani aggiornate: ", mani)
        time.sleep(1)